# Telco Customer Churn Analysis and Prediction
**AICTE | IBM SkillsBuild – Data Analytics with AI Internship 2026**

Goal: understand why customers leave a telecom company and build a model that predicts churn.

Dataset: IBM Telco Customer Churn (Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn). Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` and place it in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
RANDOM_STATE = 42

## 1. Load the data

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

## 2. Data cleaning
`TotalCharges` is stored as text because a few rows are blank (new customers with tenure 0). We convert it to a number and fill those rows with 0.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges:', df['TotalCharges'].isna().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df = df.drop(columns=['customerID'])
print('Duplicate rows:', df.duplicated().sum())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.describe()

## 3. Exploratory data analysis

In [ ]:
rate = df['Churn'].mean() * 100
print(f'Overall churn rate: {rate:.1f}%')
df['Churn'].value_counts().plot(kind='bar', color=['#4c72b0', '#c44e52'])
plt.xticks([0, 1], ['Stayed', 'Churned'], rotation=0)
plt.title('Churn distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.ravel(), ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']):
    (df.groupby(col)['Churn'].mean() * 100).plot(kind='bar', ax=ax, color='#c44e52')
    ax.set_ylabel('Churn rate (%)')
    ax.set_title(f'Churn rate by {col}')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax)
    ax.set_xticklabels(['Stayed', 'Churned'])
    ax.set_title(f'{col} vs churn')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation of numeric features')
plt.show()

## 4. Feature preparation

In [ ]:
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])
print(X_train.shape, X_test.shape)

## 5. Model training
We compare Logistic Regression and Random Forest. `class_weight='balanced'` is used because only about a quarter of customers churn.

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, class_weight='balanced'), X_train_s, X_test_s),
    'Random Forest': (RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE), X_train, X_test),
}
results, fitted = [], {}
for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    fitted[name] = (model, pred)
    results.append({'Model': name,
                    'Accuracy': accuracy_score(y_test, pred),
                    'Precision': precision_score(y_test, pred),
                    'Recall': recall_score(y_test, pred),
                    'F1': f1_score(y_test, pred),
                    'ROC-AUC': roc_auc_score(y_test, proba)})
results_df = pd.DataFrame(results).set_index('Model').round(3)
results_df

## 6. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (model, pred)) in zip(axes, fitted.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()
print(classification_report(y_test, fitted['Random Forest'][1], target_names=['Stayed', 'Churned']))

In [ ]:
rf = fitted['Random Forest'][0]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
imp[::-1].plot(kind='barh', color='#4c72b0')
plt.title('Top 10 features (Random Forest)')
plt.show()
imp

## 7. Conclusions and recommendations
- Customers on **month-to-month contracts**, with **short tenure**, **fiber optic internet** or **electronic check** payment tend to churn more (confirm with the charts above).
- Retention offers (discounts for longer contracts, onboarding support for new customers, autopay incentives) should target these groups.
- The model can rank customers by churn risk so the retention team can contact the highest-risk customers first.

**Limitations:** single dataset, no time dimension, and no cost data for retention offers. Future work: hyperparameter tuning, gradient boosting, and threshold selection based on business cost.